In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [24]:
batch_size = 128
transform = transforms.Compose([transforms.ToTensor()])

train_dataset = torchvision.datasets.MNIST(
    root='./data', train=True, download=True, transform=transform
)
test_dataset = torchvision.datasets.MNIST(
    root='./data', train=False, download=True, transform=transform
)

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 60000
Test samples: 10000


In [ ]:
def estimate_mi_binned(y, c, n_classes=10, n_bins=10, vmin=-3.0, vmax=3.0):
    """
    Approximate I(y; c) via histogram binning on y in [vmin, vmax].
    y: (batch_size,) continuous values
    c: (batch_size,) discrete labels in [0..n_classes-1]
    
    Returns negative MI (a scalar tensor), to be added to the loss (we minimize -I).
    
    NOTE: This function is not truly differentiable wrt y. 
          In normal PyTorch usage, the gradient is 0 except at bin boundaries.
    """
    batch_size = y.size(0)
    device = y.device
    
    # 1) Create bin edges
    edges = torch.linspace(vmin, vmax, n_bins+1, device=device)
    # 2) Bucketize y
    # bin_idx in [0..n_bins], but top edge means bin_idx=n_bins => clamp to n_bins-1
    bin_idx = torch.bucketize(y, edges) - 1
    bin_idx = torch.clamp(bin_idx, 0, n_bins-1)  # shape (B,)
    
    # 3) Count joint occurrences: shape (n_bins, n_classes)
    joint_counts = torch.zeros(n_bins, n_classes, device=device)
    for i in range(batch_size):
        b = bin_idx[i].item()
        cc = c[i].item()
        joint_counts[b, cc] += 1.0
    
    # 4) Convert to probabilities
    joint_prob = joint_counts / batch_size  # shape (n_bins, n_classes)
    p_bin = joint_prob.sum(dim=1, keepdim=True)  # shape (n_bins,1)
    p_c   = joint_prob.sum(dim=0, keepdim=True)  # shape (1, n_classes)
    
    # 5) Compute sum_{bin,c} p(bin,c) log [ p(bin,c) / (p(bin)*p(c)) ]
    #    We'll skip terms where p(bin,c)=0 to avoid log(0).
    mask = (joint_prob > 0)
    p_bc = joint_prob[mask]
    
    # p_b and p_c need to be aligned with the mask
    # We'll do a small trick: we find which (bin, class) indices are nonzero
    bin_idxs, class_idxs = mask.nonzero(as_tuple=True)
    p_b = p_bin[bin_idxs]
    p_cc = p_c[0, class_idxs]
    
    # log ratio
    mi_terms = p_bc * ( torch.log(p_bc) - torch.log(p_b) - torch.log(p_cc) )
    mi_value = mi_terms.sum()
    
    # return negative
    return -mi_value

In [25]:
class LocalLayerBlock(nn.Module):
    """
    A 'layer' with K scalar neurons: y_i = w_i^T x + b_i, i=1..K.
    We do NOT create separate classifiers, because we'll do binned MI directly on y_i.
    """
    def __init__(self, in_dim, K=10):
        super().__init__()
        # Weights: map from in_dim -> K scalars
        self.linear = nn.Linear(in_dim, K, bias=True)
    
    def forward(self, x):
        # x: (B, in_dim)
        # returns Y: (B, K) => each column Y[:,i] is the output of neuron i
        Y = self.linear(x)
        return Y
    
    def redundancy_penalty(self):
        """
        Sum of dot products w_i dot w_j for i<j.
        We'll treat the weight matrix shape as (K, in_dim) in PyTorch's ordering.
        """
        W = self.linear.weight  # (K, in_dim)
        # dot_mat[i,j] = w_i dot w_j
        dot_mat = torch.matmul(W, W.t())  # shape (K, K)
        
        K_ = dot_mat.shape[0]
        off_diag_sum = 0.0
        for i in range(K_):
            for j in range(i+1, K_):
                off_diag_sum += dot_mat[i, j]
        return off_diag_sum

In [26]:
def train_local_layer_binned_mi(
    layer_block,
    train_loader,
    lr=1e-3,
    lambda_reg=0.01,
    epochs=3,
    n_bins=10,
    device='cpu'
):
    """
    Train the layer with cost = sum_i [-I(y_i;c)] + lambda * redundancy
    using naive binned MI for each neuron.
    
    NOTE: This is typically non-differentiable -> expect no or minimal improvement.
    """
    layer_block.to(device)
    optimizer = optim.Adam(layer_block.parameters(), lr=lr)
    
    for epoch in range(epochs):
        layer_block.train()
        total_loss = 0.0
        
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            Y = layer_block(images)  # shape (B, K)
            
            # sum binned-MI across neurons
            mi_sum = 0.0
            K_ = Y.shape[1]
            for i in range(K_):
                y_i = Y[:, i]
                # -I(y_i; c)
                neg_mi_i = estimate_mi_binned(y_i, labels, n_classes=10, n_bins=n_bins)
                mi_sum += neg_mi_i
            
            # redundancy
            red_pen = layer_block.redundancy_penalty()
            
            # total cost
            loss = mi_sum + lambda_reg * red_pen
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        avg_loss = total_loss / len(train_loader)
        print(f"Epoch [{epoch+1}/{epochs}] - Local Layer Loss: {avg_loss:.4f}")
    
    return layer_block

In [27]:
def freeze_module(module):
    for param in module.parameters():
        param.requires_grad = False

def get_hidden_representation(layer_block, data_loader):
    layer_block.eval()
    all_feats = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1).to(device)
            Y = layer_block(images)  # (B, K)
            all_feats.append(Y.cpu())
            all_labels.append(labels)
    return torch.cat(all_feats, dim=0), torch.cat(all_labels, dim=0)

def build_and_train_2layer_binned_mi(train_loader, test_loader, device=device):
    # 1) Train first layer with binned MI
    layer1 = LocalLayerBlock(in_dim=784, K=16)
    train_local_layer_binned_mi(layer1, train_loader, lr=1e-3, lambda_reg=0.01, epochs=3, n_bins=10, device=device)
    
    # 2) Freeze layer1
    freeze_module(layer1)
    
    # 3) Extract hidden
    X_train_hid, y_train = get_hidden_representation(layer1, train_loader)
    X_test_hid, y_test = get_hidden_representation(layer1, test_loader)
    
    # Build new loaders for second layer
    train_hidden_dataset = torch.utils.data.TensorDataset(X_train_hid, y_train)
    test_hidden_dataset = torch.utils.data.TensorDataset(X_test_hid, y_test)
    train_hidden_loader = torch.utils.data.DataLoader(train_hidden_dataset, batch_size=128, shuffle=True)
    test_hidden_loader  = torch.utils.data.DataLoader(test_hidden_dataset, batch_size=128, shuffle=False)
    
    # 4) Train second layer
    layer2 = LocalLayerBlock(in_dim=16, K=16)
    train_local_layer_binned_mi(layer2, train_hidden_loader, lr=1e-3, lambda_reg=0.01, epochs=3, n_bins=10, device=device)
    
    # Return references
    return layer1, layer2, train_hidden_loader, test_hidden_loader

In [28]:
def evaluate_second_layer(layer2, data_loader, device='cpu'):
    """
    Evaluate classification by averaging the logits from all K neurons in layer2.
    """
    layer2.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for X_hid, labels in data_loader:
            X_hid = X_hid.to(device)
            labels = labels.to(device)
            
            Y, logits_list = layer2(X_hid)
            # logits_list is a list of K => shape is (B,10) for each
            stacked_logits = torch.stack(logits_list, dim=0)  # (K, B, 10)
            avg_logits = torch.mean(stacked_logits, dim=0)    # (B, 10)
            
            preds = avg_logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    return 100.0 * correct / total

In [29]:
layer1, layer2, train_hidden_loader, test_hidden_loader = build_and_train_2layer_local(train_loader, test_loader, device=device)

# Evaluate accuracy on test set (using the second layer's ensemble of neurons).
test_acc_local = evaluate_second_layer(layer2, test_hidden_loader, device=device)
print(f"Final 2-layer local learning test accuracy: {test_acc_local:.2f}%")

Epoch [1/3] - Local Layer Loss: 29.4524
Epoch [2/3] - Local Layer Loss: 1.5201
Epoch [3/3] - Local Layer Loss: -51.8815
Epoch [1/3] - Local Layer Loss: 36.5483
Epoch [2/3] - Local Layer Loss: 29.8279
Epoch [3/3] - Local Layer Loss: 28.1628
Final 2-layer local learning test accuracy: 78.28%


In [30]:
class TwoLayerNet(nn.Module):
    def __init__(self, in_dim=784, hidden_dim=64, num_classes=10):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)
    
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [31]:
def train_backprop(model, train_loader, lr=1e-3, epochs=5, device='cpu'):
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    ce_loss = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for images, labels in train_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            
            optimizer.zero_grad()
            logits = model(images)
            loss = ce_loss(logits, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {running_loss/len(train_loader):.4f}")

def evaluate_accuracy(model, data_loader, device='cpu'):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)
            logits = model(images)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total

In [32]:
model_bp = TwoLayerNet(in_dim=784, hidden_dim=64, num_classes=10)
train_backprop(model_bp, train_loader, lr=1e-3, epochs=5, device=device)

test_acc_bp = evaluate_accuracy(model_bp, test_loader, device=device)
print(f"Standard Backprop 2-layer test accuracy: {test_acc_bp:.2f}%")

Epoch [1/5] - Loss: 0.4830
Epoch [2/5] - Loss: 0.2404
Epoch [3/5] - Loss: 0.1882
Epoch [4/5] - Loss: 0.1549
Epoch [5/5] - Loss: 0.1327
Standard Backprop 2-layer test accuracy: 95.89%
